In [0]:
Select *
From brighttv_data.data.user_profile;

Select * 
From brighttv_data.data.viewership;

SELECT *
From brighttv_data.data.user_profile
limit 100;
-------------------------
--checking duplicates
------------------------
Select count(*), userid
from brighttv_data.data.user_profile
group by UserID
having count(*)>1;

------------------------
--Gender checks
------------------------
select distinct gender
from brighttv_data.data.user_profile;
--------------------------------------
--cleaning gender column
--------------------------------------
SELECT DISTINCT
 CASE 
    WHEN gender = 'None' THEN 'unknown'
    WHEN gender = ' ' THEN 'unknown'
    WHEN gender IS NULL THEN 'unknown' 
    ELSE gender
 END AS sex
FROM brighttv_data.data.user_profile;

--------------------------------------------
-- Race Checks
--------------------------------------------
SELECT DISTINCT Race
from brighttv_data.data.user_profile;

SELECT COUNT(DISTINCT userid)  AS subs,
        CASE 
            WHEN race = 'other' THEN 'unknown' -- Replace other with unknown
            WHEN race = 'None' THEN 'unknown' -- Replace None with unknown
            WHEN race = ' ' THEN 'unknown' -- Replace empty with unknown
            WHEN race IS NULL THEN 'unknown' -- Replace null with unknown
        ELSE race -- keep it as it r
        END AS enthnicity -- new column 
FROM brighttv_data.data.user_profile
GROUP BY enthnicity;    

--------------------------------------------
-- Province Checks
--------------------------------------------

SELECT DISTINCT Province
FROM brighttv_data.data.user_profile;

SELECT DISTINCT 
 CASE 
    WHEN Province = 'None' THEN 'unknown'
    WHEN Province =' ' THEN 'unknown'
    WHEN Province IS NULL THEN 'unknown'
    ELSE Province
    END AS Region
from brighttv_data.data.user_profile;

--------------------------------------------
-- Age Checks
--------------------------------------------
SELECT MIN(Age) AS min_age, -- check the youngest person
       MAX(Age) AS max_age, -- find the oldest person
       AVG(Age) AS mean_age 
FROM brighttv_data.data.user_profile;

SELECT 
    CASE
        WHEN Age = 0 THEN 'Infant'
        WHEN Age BETWEEN 1 AND 12 THEN 'Kids'
        WHEN Age BETWEEN 13 AND 17 THEN 'Youth'
        WHEN Age BETWEEN 18 AND 35 THEN 'Young Adult'
        WHEN Age BETWEEN 36 AND 50 THEN 'Adults'
        WHEN Age > 50 AND AGE<=60 THEN 'Elder'
        WHEN AGE>60 THEN 'Pensioner'
    END AS Age_group
from brighttv_data.data.user_profile;     

SELECT MIN(Age) AS min_age, --- = 0
        MAX(Age) AS max_age -- = 114
FROM brighttv_data.data.user_profile;

SELECT COUNT(*) AS cnt
FROM brighttv_data.data.user_profile
WHERE age IS NULL;

WITH user_profiles AS (
SELECT UserID,
  CASE 
        WHEN Province=' ' THEN 'Uncategorized'
        WHEN Province='None' THEN 'Uncategorized'
    ELSE Province
    END AS Region,
    age,
    CASE
        WHEN age = 0 THEN 'Infants'
        WHEN age BETWEEN 1 AND 12 THEN 'Kids'
        WHEN age BETWEEN 13 AND 19 THEN 'Teenager'
        WHEN age BETWEEN 20 AND 35 THEN 'Youth'
        WHEN age BETWEEN 36 AND 50 THEN 'Adult'
        WHEN age BETWEEN 51 AND 65 THEN 'Elder'
        WHEN age >65 THEN 'Pensioner'
    END AS age_groups,
    CASE
        WHEN (email IS NOT NULL )OR (email=' ') OR  (email NOT IN ('None'))THEN 1
    ELSE 0
    END AS email_flag,
    CASE
        WHEN `Social Media Handle` IS NOT NULL OR `Social Media Handle`=' ' OR  `Social Media Handle` NOT IN ('None')THEN 1
        ELSE 0
    END AS sm_flag,
    CASE
        WHEN Race='other' THEN 'None'
        WHEN Race=' ' THEN 'None'
    ELSE Race
    END AS Race,
    CASE
        WHEN gender =' ' THEN 'None'
        ELSE gender
    END AS Gender
from brighttv_data.data.user_profile
),
viewership AS (
    SELECT
    COALESCE(UserID0,userid4) AS userid,
    TO_CHAR(RecordDate2, 'yyyyMM') AS month_id,
    TO_DATE(RecordDate2) AS watch_date,
    --TIME(RecordDate2) AS watch_time,
    TO_CHAR(RecordDate2, 'DD') AS day_of_week,
    DAYNAME(RecordDate2) AS day_name,
CASE
        WHEN day_name IN ('Sat', 'Sun') THEN 'weekend'
        ELSE 'weekday'
    END AS day_classification,
    MONTHNAME(RecordDate2) AS month_name,
    CASE 
        WHEN Channel2 IN ('SawSee','Sawsee') THEN 'SawSee'
        WHEN Channel2 IN ('SuperSport Live Events','Live on SuperSport', 'Supersport Live Events', 'DStv Events 1') THEN 'Live Events'
    ELSE Channel2
    END AS Tv_channel,
date_format(RecordDate2, 'HH:mm:ss') AS watch_time,
    CASE
        WHEN watch_time BETWEEN '00:00:00' AND '05:59:59' THEN '01. Midnight'
        WHEN watch_time BETWEEN '06:00:00' AND '11:59:59' THEN '02. Morning'
        WHEN watch_time BETWEEN '12:00:00' AND '16:59:59' THEN '03. Afternoon'
        WHEN watch_time BETWEEN '17:00:00' AND '23:59:59' THEN '04. Evening'
    END AS time_of_day,
DATE_FORMAT(`Duration 2`, 'HH:mm:ss') AS duration,
    CASE 
        WHEN duration BETWEEN '00:05:00' AND '00:30:00' THEN '01. Low Usage: <30 min'
        WHEN duration BETWEEN '00:30:01' AND '00:59:59' THEN '02. Med Usage: <60 min'
        WHEN duration > '00:59:59' THEN '03. High Usage: >60 min'
        ELSE '04. No Usage'
    END AS screen_time_bucket,
    HOUR(RecordDate2) AS hour_of_day
    from brighttv_data.data.viewership
)
SELECT Coalesce(A.userid,B.userid) AS sub_id,
       month_id,
       watch_date,
       day_of_week,
       day_name,
       day_classification,
       month_name,
       Tv_channel,
       time_of_day,
       hour_of_day,
       screen_time_bucket,
       --user_flag,
       duration,
       Region,
       age_groups,
       email_flag,
       sm_flag,
       Race,
       Gender
FROM viewership AS A
LEFT JOIN user_profiles AS B
ON A.userid=B.userid;



In [0]:
-- Preview viewership table
SELECT * 
FROM brighttv_data.data.viewership
LIMIT 100;

In [0]:
-- Check for duplicate UserIDs
SELECT COUNT(*) AS duplicate_count, UserID
FROM brighttv_data.data.user_profile
GROUP BY UserID
HAVING COUNT(*) > 1;

In [0]:
-- Check distinct gender values
SELECT DISTINCT Gender
FROM brighttv_data.data.user_profile
ORDER BY Gender;